# Numerical Integration and Convergence

## A test problem with a known answer

Numerical integration replaces a continuous area by a finite calculation. To determine whether an algorithm is working, we begin with a function whose exact integral is known:

$$f(x)=\frac{3}{2}(1-x^2), \qquad 0\leq x\leq1.$$

Direct integration gives

$$I=\int_0^1f(x)\,dx
=\frac{3}{2}\left[x-\frac{x^3}{3}\right]_0^1=1.$$

This exact value lets us measure the numerical error

$$E_N=|I_N-I|,$$

where $I_N$ is an estimate constructed with $N$ subintervals or samples. The central question is not just whether an estimate is close to 1, but **how rapidly it approaches 1 as computational effort increases**.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt


def integrand(x):
    '''Test function with an exact integral of 1 on [0, 1].'''
    return 1.5*(1.0-x**2)


xlow = 0.0
xhigh = 1.0
exact_integral = 1.0
sample_counts = 10**np.arange(2, 7)  # 10^2 through 10^6

## Composite midpoint rule

Divide $[a,b]$ into $N$ equal subintervals of width

$$h=\frac{b-a}{N}.$$

The midpoint of subinterval $i$ is $x_i^*=a+(i+1/2)h$, and the composite midpoint estimate is

$$I_N^{(M)}=h\sum_{i=0}^{N-1}f(x_i^*).$$

For a sufficiently smooth function, its global error is

$$E_N^{(M)}=O(h^2)=O(N^{-2}).$$

Therefore increasing $N$ by a factor of ten should reduce the error by approximately a factor of one hundred. On a log–log plot of error versus $N$, we expect a slope of $-2$ until floating-point roundoff becomes important.

In [ ]:
midpoint_estimates = []

for n_intervals in sample_counts:
    h = (xhigh-xlow)/n_intervals
    midpoints = xlow+(np.arange(n_intervals)+0.5)*h
    estimate = h*np.sum(integrand(midpoints))
    midpoint_estimates.append(estimate)

midpoint_estimates = np.asarray(midpoint_estimates)
midpoint_errors = np.abs(midpoint_estimates-exact_integral)

for n_intervals, estimate, error in zip(
        sample_counts, midpoint_estimates, midpoint_errors):
    print(f"N={n_intervals:7d}: I_N={estimate:.15f}, absolute error={error:.3e}")

## Estimate and convergence plots

The first plot compares each estimate with the exact value. The second plot is more diagnostic: it tests the predicted power law. A line proportional to $N^{-2}$ is anchored to the first measured error so that its slope, rather than its arbitrary vertical normalization, can be compared with the data.

The plotted vertical differences are **absolute numerical errors**, not measurement uncertainties and therefore not error bars.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].plot(sample_counts, midpoint_estimates, 'o-', label='Midpoint estimate')
axes[0].axhline(exact_integral, color='k', linestyle='--', label='Exact integral')
axes[0].set_xscale('log')
axes[0].set_xlabel('Number of subintervals, N')
axes[0].set_ylabel('Integral estimate')
axes[0].set_title('Composite Midpoint Estimates')
axes[0].legend()

reference = midpoint_errors[0]*(sample_counts/sample_counts[0])**(-2.0)
axes[1].loglog(sample_counts, midpoint_errors, 'o-', label='Absolute error')
axes[1].loglog(sample_counts, reference, '--', label=r'Reference: $N^{-2}$')
axes[1].set_xlabel('Number of subintervals, N')
axes[1].set_ylabel(r'$|I_N-I|$')
axes[1].set_title('Midpoint Convergence')
axes[1].legend()

slope = np.polyfit(np.log10(sample_counts), np.log10(midpoint_errors), 1)[0]
print(f"Measured log-log slope = {slope:.4f} (expected approximately -2)")

plt.tight_layout()
plt.show()

## Interpretation

The observed slope should be close to $-2$, confirming second-order convergence. Eventually the truncation error becomes comparable to floating-point accumulation and roundoff error; beyond that point, increasing $N$ need not improve the answer. This competition between truncation and roundoff is why “more intervals” is not an unlimited guarantee of greater accuracy.